# EDA — NYC Airbnb

Loads the raw sample from W&B, runs a quick profile, applies the same cleaning logic as the `basic_cleaning` step (driven by `config.yaml`), then profiles the cleaned result.

In [ ]:
import pandas as pd
import yaml
from pathlib import Path

import wandb
from ydata_profiling import ProfileReport

In [ ]:
config = yaml.safe_load((Path("../../config.yaml")).read_text())

PROJECT_NAME         = config["main"]["project_name"]
GROUP_NAME           = config["etl"]["group"]
JOB_TYPE             = config["etl"]["eda"]["job_type"]
SAMPLE_ARTIFACT      = config["etl"]["download"]["output_artifact"]
SAMPLE_ARTIFACT_TYPE = config["etl"]["download"]["output_type"]

run = wandb.init(project=PROJECT_NAME, group=GROUP_NAME, job_type=JOB_TYPE, save_code=True)

## Raw Data Profile

In [ ]:
artifact = run.use_artifact(f"{SAMPLE_ARTIFACT}:latest", type=SAMPLE_ARTIFACT_TYPE)
artifact_path = artifact.file()

df_raw = pd.read_csv(artifact_path)
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
profile_raw = ProfileReport(df_raw, title=f"{PROJECT_NAME} — Raw Data", explorative=True)
profile_raw.to_notebook_iframe()

## Data Cleaning

Mirrors `src/basic_cleaning/run.py`: applies the filters from `config.yaml`, fills missing `reviews_per_month`, and parses `last_review` as a datetime.

In [ ]:
import pandas as pd

df = df_raw.copy()

for column, low, high in config["etl"]["basic_cleaning"]["filters"]:
    before = len(df)
    mask = pd.Series(True, index=df.index)
    if low is not None:
        mask &= df[column] >= low
    if high is not None:
        mask &= df[column] <= high
    df = df[mask]
    print(f"Filtered {column} [{low}, {high}]: {before - len(df)} rows dropped")

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
df["last_review"] = pd.to_datetime(df["last_review"])

print(f"\nFinal shape: {df.shape}")
df.head()

## Cleaned Data Profile

In [ ]:
profile_clean = ProfileReport(df, title=f"{PROJECT_NAME} — Cleaned Data", explorative=True)
profile_clean.to_notebook_iframe()

run.finish()